# Pair-grid apparatus — location analysis T2 (prompt 28)

The 8c apparatus (pi_0911 §8c + prompt-28 pinned refinements) run pair-generically on the
3 priority pair grids (**nuclear×wind_317, pv×tail, nuclear×pv** — 81/81 rows each, collected
by pairgrid-bot) plus the existing **wind_303×wind_317** (`contour_303x317_C`, re-estimated
under the same pinned estimators for continuity).

**Estimation discipline (pinned):** verdict statistics (tot/full/Δloc/tilt, I) are fits to the
raw 81 points — no interpolation enters them. Along-line quantities come from the GP posterior
mean (Matérn-2.5 ARD, per-fold standardization — the 19b apparatus), gated by LOOCV RMSE ≤
floor; the two floorless objectives (reserve, starts) get NO along-line quantitative reads.
Bilinear contours are display-only. **Nuclear rows carry the g1 fuel-deletion APPROX caveat
(patch PROPOSED — `analysis/quality_report/`, pending PI review — NOT implemented here).**

In [1]:
import json, os, sys, itertools, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"axes.labelweight": "bold", "axes.titleweight": "bold",
                     "font.weight": "bold"})
from scipy import stats

CAMPAIGN = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, CAMPAIGN)
from tiers import TIERS, STAGE2_LATTICE, load_gen_pmax

OBJ_COLS = ["load_shed_mwh", "true_curtailment_mwh", "total_cost_raw_usd",
            "total_cost_less_synthetic_usd", "reserve_shortfall_mwh", "thermal_starts"]
FLOORS = {"load_shed_mwh": 3000.0, "true_curtailment_mwh": 5000.0,
          "total_cost_raw_usd": 0.5e6, "total_cost_less_synthetic_usd": 0.5e6,
          "reserve_shortfall_mwh": None, "thermal_starts": None}   # pi_0911 §3.5
FLOORED = [o for o in OBJ_COLS if FLOORS[o] is not None]
SHORT = {"load_shed_mwh": "shed", "true_curtailment_mwh": "curt",
         "total_cost_raw_usd": "cost_raw", "total_cost_less_synthetic_usd": "cost_ls",
         "reserve_shortfall_mwh": "reserve", "thermal_starts": "starts"}

_PMAX = load_gen_pmax()
NAMEPLATE = {t: sum(_PMAX[m] for m in TIERS[t]["members"]) for t in TIERS}
LAT = np.round(np.asarray(STAGE2_LATTICE, float), 10)
MC_SEED = 20260920
rng = np.random.default_rng(MC_SEED)
_Q = {n: float(np.percentile(np.ptp(rng.normal(size=(10_000, n)), axis=1), 95))
      for n in range(2, 20)}   # 95th-pct range factors, floor ≡ 1σ Gaussian

PAIRS = [("nuclear", "wind_317", "pairgrid_nuclear_wind_317_C"),
         ("pv", "tail", "pairgrid_pv_tail_C"),
         ("nuclear", "pv", "pairgrid_nuclear_pv_C"),
         ("wind_303", "wind_317", "contour_303x317_C")]

def load_pair(a, b, wave):
    wdir = os.path.join(CAMPAIGN, "waves", wave)
    if wave.startswith("pairgrid"):
        man = json.load(open(os.path.join(wdir, "manifest.json")))
        assert man["pair"] == [a, b], (wave, man["pair"])   # manifest is the authority
    dm = pd.read_csv(os.path.join(wdir, "design_matrix.csv"))
    ob = pd.read_csv(os.path.join(wdir, "objectives.csv"))
    df = dm.merge(ob[["index"] + OBJ_COLS], on="index", validate="1:1")
    # round to 10 decimals before ANY omega use — the old contour CSV stores
    # raw-linspace doubles (exact == fails at one level)
    df["wa"] = df[f"{a}_omega"].round(10); df["wb"] = df[f"{b}_omega"].round(10)
    assert df["wa"].isin(LAT).all() and df["wb"].isin(LAT).all(), wave
    others = [t for t in TIERS if t not in (a, b)]
    assert df[[f"{t}_omega" for t in others]].isna().all().all(), wave
    assert len(df) == 81 and len(df[["wa", "wb"]].drop_duplicates()) == 81
    df["x"] = df["wa"] * NAMEPLATE[a]; df["y"] = df["wb"] * NAMEPLATE[b]
    return df.sort_values(["wa", "wb"]).reset_index(drop=True)

DATA = {(a, b): load_pair(a, b, w) for a, b, w in PAIRS}
for (a, b), df in DATA.items():
    print(f"{a} x {b}: 81 rows OK; MW rectangle [{df.x.min():.1f},{df.x.max():.1f}] x "
          f"[{df.y.min():.1f},{df.y.max():.1f}]")

# scenario-C no-PEM base = the absolute anchor for physical-units panels,
# recovered from sweep_C's delta columns (constant across rows -> verified)
swC = pd.read_csv(os.path.join(CAMPAIGN, "waves", "sweep_C", "objectives.csv"))
BASE = {}
for obj, dcol in (("true_curtailment_mwh", "delta_curtailment_mwh"),
                  ("load_shed_mwh", "delta_load_shed_mwh"),
                  ("total_cost_less_synthetic_usd", "delta_cost_less_synthetic_usd_APPROX"),
                  ("reserve_shortfall_mwh", "delta_reserve_shortfall_mwh"),
                  ("thermal_starts", "delta_thermal_starts")):
    bases = swC[obj] - swC[dcol]
    assert bases.std() < 1e-6 * max(1.0, abs(bases.mean())), obj
    BASE[obj] = float(bases.iloc[0])
BASE["total_cost_raw_usd"] = BASE["total_cost_less_synthetic_usd"]   # base has no synthetic
print("base anchors:", {SHORT[k]: round(v, 1) for k, v in BASE.items()})

nuclear x wind_317: 81 rows OK; MW rectangle [20.0,400.0] x [40.0,799.1]
pv x tail: 81 rows OK; MW rectangle [17.0,340.5] x [12.6,251.5]
nuclear x pv: 81 rows OK; MW rectangle [20.0,400.0] x [17.0,340.5]
wind_303 x wind_317: 81 rows OK; MW rectangle [42.4,847.0] x [40.0,799.1]
base anchors: {'curt': 1688575.8, 'shed': 41500.1, 'cost_ls': 522480773.8, 'reserve': 203681.4, 'starts': 5183.0, 'cost_raw': 522480773.8}


## Verdict statistics on the raw 81 points (per pair × objective)

- **tot** = R² of a quadratic in x+y (3 params); **full** = R² of the full 2-D quadratic
  (6 params); **Δloc = full − tot** (reported for 8c continuity, never gated on);
  the location-effect **gate** is `extra-RMSE = √((SSE_tot − SSE_full)/81)` vs the floor.
- **tilt** = slope(tierB)/slope(tierA) from the planar fit (sorted tier order), reported as
  log₂ tilt with a 95% CI from floor-propagated coefficient SEs (σ²(AᵀA)⁻¹, σ ≡ floor);
  *tilted* requires the CI to exclude 1 AND planar dominance (reading suppressed where
  quadratic-over-planar extra-RMSE exceeds the floor). Floorless objectives: point estimates
  only, no CI, no gates — range-and-replication governs their verdicts.
- **I_raw** = f(1,1) − f(1,0.05) − f(0.05,1) + f(0.05,0.05), all four corners from the SAME
  grid; floor = 2× the per-value floor. **I_rel** = I_raw / |Σ within-grid main effects|,
  reported only where the denominator ≥ 3× floor.

In [2]:
def corner(df, wa, wb, obj):
    r = df[(df.wa == wa) & (df.wb == wb)]
    assert len(r) == 1
    return float(r[obj].iloc[0])

def pair_stats(df, a, b, obj):
    x, y, z = df.x.to_numpy(), df.y.to_numpy(), df[obj].to_numpy(float)
    n = len(z); sst = float(((z - z.mean()) ** 2).sum())
    # planar fit [x, y, 1] in MW
    A1 = np.c_[x, y, np.ones(n)]
    beta1, *_ = np.linalg.lstsq(A1, z, rcond=None)
    sse1 = float(((z - A1 @ beta1) ** 2).sum())
    sa, sb = beta1[0], beta1[1]
    # full 2-D quadratic (6 params) and total-only quadratic (3 params)
    Aq = np.c_[np.ones(n), x, y, x**2, y**2, x*y]
    betaq, *_ = np.linalg.lstsq(Aq, z, rcond=None)
    sseq = float(((z - Aq @ betaq) ** 2).sum())
    tot = x + y
    At = np.c_[np.ones(n), tot, tot**2]
    betat, *_ = np.linalg.lstsq(At, z, rcond=None)
    sset = float(((z - At @ betat) ** 2).sum())
    out = {"pair": f"{a}|{b}", "objective": obj,
           "r2_tot": 1 - sset / sst, "r2_full": 1 - sseq / sst}
    out["dloc"] = out["r2_full"] - out["r2_tot"]
    out["extra_rmse_loc"] = float(np.sqrt(max(sset - sseq, 0.0) / n))
    out["extra_rmse_qp"] = float(np.sqrt(max(sse1 - sseq, 0.0) / n))   # quadratic over planar
    out["tilt"] = sb / sa if abs(sa) > 1e-12 else np.nan
    out["log2_tilt"] = float(np.log2(out["tilt"])) if out["tilt"] and out["tilt"] > 0 else np.nan
    floor = FLOORS[obj]
    if floor is not None:
        cov = floor**2 * np.linalg.inv(A1.T @ A1)
        va, vb, cab = cov[0, 0], cov[1, 1], cov[0, 1]
        if abs(sa) > 1e-12 and abs(sb) > 1e-12:
            var_ln = va/sa**2 + vb/sb**2 - 2*cab/(sa*sb)
            half = 1.96 * np.sqrt(max(var_ln, 0.0)) / np.log(2)
            out["log2_tilt_lo"], out["log2_tilt_hi"] = out["log2_tilt"] - half, out["log2_tilt"] + half
        out["loc_gate_ratio"] = out["extra_rmse_loc"] / floor
        out["planar_dominates"] = bool(out["extra_rmse_qp"] <= floor)
        out["tilted_verdict"] = bool(out.get("log2_tilt_lo", np.nan) > 0 or
                                     out.get("log2_tilt_hi", np.nan) < 0) and out["planar_dominates"]
    # pinned interaction index (corners in omega space)
    f11 = corner(df, 1.0, 1.0, obj); f10 = corner(df, 1.0, 0.05, obj)
    f01 = corner(df, 0.05, 1.0, obj); f00 = corner(df, 0.05, 0.05, obj)
    out["I_raw"] = f11 - f10 - f01 + f00
    denom = abs((f10 - f00) + (f01 - f00))
    out["I_denom"] = denom
    if floor is not None:
        out["I_raw_over_floor"] = out["I_raw"] / (2 * floor)
        out["I_rel"] = out["I_raw"] / denom if denom >= 3 * floor else np.nan
    return out

VS = pd.DataFrame([pair_stats(DATA[(a, b)], a, b, obj)
                   for (a, b) in DATA for obj in OBJ_COLS])
VS.to_csv("pair_verdict_stats.csv", index=False)
show = ["pair", "objective", "r2_tot", "r2_full", "dloc", "loc_gate_ratio",
        "log2_tilt", "log2_tilt_lo", "log2_tilt_hi", "tilted_verdict",
        "I_raw", "I_raw_over_floor", "I_rel"]
with pd.option_context("display.width", 220, "display.max_columns", 30):
    print(VS[[c for c in show if c in VS]].round(3).to_string(index=False))

             pair                     objective  r2_tot  r2_full  dloc  loc_gate_ratio  log2_tilt  log2_tilt_lo  log2_tilt_hi tilted_verdict       I_raw  I_raw_over_floor  I_rel
 nuclear|wind_317                 load_shed_mwh   0.930    0.985 0.055           0.762      0.845         0.452         1.238           True   10846.235             1.808  0.242
 nuclear|wind_317          true_curtailment_mwh   0.953    0.998 0.045           8.623     -0.670        -0.687        -0.654          False  259620.648            25.962  0.219
 nuclear|wind_317            total_cost_raw_usd   0.566    1.000 0.434          30.543     -2.284        -2.303        -2.264          False 3799286.891             3.799  0.043
 nuclear|wind_317 total_cost_less_synthetic_usd   0.575    0.999 0.424          29.248     -2.243        -2.262        -2.223          False 8499918.810             8.500  0.106
 nuclear|wind_317         reserve_shortfall_mwh   0.947    0.989 0.043             NaN      0.139           Na

## GP gate + along-line reads (floored objectives only)

Matérn-2.5 ARD per the 19b reference (`bo_replay/replay.py:182-199`): per-fold min-max input
scaling + `normalize_y`; LOOCV RMSE ≤ floor gates any along-line read. Along each iso-total
guide (clipped to the sampled rectangle) the max−min of the GP mean is a range statistic —
floored by the MC range floor with n = the number of grid cells the segment traverses.

In [3]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.exceptions import ConvergenceWarning

def base_kernel(d):
    return (ConstantKernel(1.0, (1e-3, 1e3))
            * Matern(length_scale=np.ones(d), length_scale_bounds=(1e-2, 1e1), nu=2.5)
            + WhiteKernel(1e-4, (1e-6, 1e0)))

def fit_gp(X, y, kernel=None, restarts=2):
    lo = X.min(axis=0); span = np.where((X.max(0) - lo) <= 0, 1.0, X.max(0) - lo)
    g = GaussianProcessRegressor(kernel=kernel or base_kernel(X.shape[1]),
                                 normalize_y=True, n_restarts_optimizer=restarts,
                                 random_state=0, alpha=1e-8)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        g.fit((X - lo) / span, y)
    return g, lo, span

def loo_rmse(X, y):
    gfull, _, _ = fit_gp(X, y)                       # hyperparams once, warm-start folds
    k0 = gfull.kernel_
    preds = np.empty(len(y))
    for i in range(len(y)):
        m = np.arange(len(y)) != i
        g, lo, span = fit_gp(X[m], y[m], kernel=k0, restarts=0)
        preds[i] = g.predict(((X[i] - lo) / span).reshape(1, -1))[0]
    return float(np.sqrt(np.mean((preds - y) ** 2))), gfull

GP, gate_rows = {}, []
for (a, b), df in DATA.items():
    X = df[["x", "y"]].to_numpy(float)
    for obj in FLOORED:
        rmse, gfull = loo_rmse(X, df[obj].to_numpy(float))
        GP[(a, b, obj)] = (gfull, X.min(0), np.where((X.max(0)-X.min(0))<=0,1,X.max(0)-X.min(0)))
        gate_rows.append({"pair": f"{a}|{b}", "objective": obj, "loocv_rmse": rmse,
                          "floor": FLOORS[obj], "gate_pass": bool(rmse <= FLOORS[obj])})
GATE = pd.DataFrame(gate_rows)
GATE.to_csv("pair_gp_gate.csv", index=False)
print(GATE.round(1).to_string(index=False))

             pair                     objective  loocv_rmse    floor  gate_pass
 nuclear|wind_317                 load_shed_mwh      1357.9   3000.0       True
 nuclear|wind_317          true_curtailment_mwh      6369.5   5000.0      False
 nuclear|wind_317            total_cost_raw_usd    262943.8 500000.0       True
 nuclear|wind_317 total_cost_less_synthetic_usd    339860.0 500000.0       True
          pv|tail                 load_shed_mwh      1174.0   3000.0       True
          pv|tail          true_curtailment_mwh      6331.8   5000.0      False
          pv|tail            total_cost_raw_usd    242431.6 500000.0       True
          pv|tail total_cost_less_synthetic_usd    237237.4 500000.0       True
       nuclear|pv                 load_shed_mwh      1690.9   3000.0       True
       nuclear|pv          true_curtailment_mwh      5169.6   5000.0      False
       nuclear|pv            total_cost_raw_usd    212954.6 500000.0       True
       nuclear|pv total_cost_less_synthe

In [4]:
def clip_segment(T, xmin, xmax, ymin, ymax):
    # segment x+y=T restricted to the rectangle; None if no intersection
    x1, x2 = max(xmin, T - ymax), min(xmax, T - ymin)
    if x2 <= x1:
        return None
    return x1, x2

line_rows = []
for (a, b), df in DATA.items():
    xmin, xmax, ymin, ymax = df.x.min(), df.x.max(), df.y.min(), df.y.max()
    cell_x = (xmax - xmin) / 8; cell_y = (ymax - ymin) / 8
    Ts = np.percentile(df.x + df.y, [20, 35, 50, 65, 80])
    for obj in FLOORED:
        gate = GATE[(GATE.pair == f"{a}|{b}") & (GATE.objective == obj)].gate_pass.iloc[0]
        g, lo, span = GP[(a, b, obj)]
        for T in Ts:
            seg = clip_segment(T, xmin, xmax, ymin, ymax)
            if seg is None:
                continue
            xs = np.linspace(seg[0], seg[1], 200); ys = T - xs
            n_cells = int(np.ceil((seg[1] - seg[0]) / cell_x) +
                          np.ceil((seg[1] - seg[0]) / cell_y))   # cells traversed
            n_cells = int(np.clip(n_cells, 2, 19))
            if not gate:
                line_rows.append({"pair": f"{a}|{b}", "objective": obj, "T_mw": T,
                                  "read": "SUPPRESSED (LOOCV gate fail)"})
                continue
            mu = g.predict((np.c_[xs, ys] - lo) / span)
            rangev = float(mu.max() - mu.min())
            fl = FLOORS[obj] * _Q[n_cells]
            line_rows.append({"pair": f"{a}|{b}", "objective": obj, "T_mw": float(T),
                              "gp_range": rangev, "range_floor": fl,
                              "ratio": rangev / fl, "n_cells": n_cells, "read": "ok"})
LINES = pd.DataFrame(line_rows)
LINES.to_csv("pair_isototal_reads.csv", index=False)
ok = LINES[LINES.read == "ok"]
piv = ok.pivot_table(index="pair", columns="objective", values="ratio", aggfunc="median")
print("median along-line (max−min)/range-floor per pair (gated reads only):")
print(piv.round(2).to_string())
print("\nsuppressed:", len(LINES) - len(ok), "reads (gate failures)")

median along-line (max−min)/range-floor per pair (gated reads only):
objective          load_shed_mwh  total_cost_less_synthetic_usd  total_cost_raw_usd
pair                                                                               
nuclear|pv                  0.42                          14.00               15.57
nuclear|wind_317            0.43                          21.47               22.48
pv|tail                     0.10                           1.26                1.27
wind_303|wind_317           0.76                           4.20                4.70

suppressed: 20 reads (gate failures)


## 8c panels (one figure per pair)

Contours = linear interpolation of the raw 9×9 (display-only), MW axes; dashed grey iso-total
guides clipped to the rectangle; dots = the 81 runs. Panels whose 7-level contour spacing is
below the floor are greyed with the annotation, not read. Nuclear pairs: g1 APPROX caveat.

In [5]:
for (a, b), df in DATA.items():
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    xmin, xmax, ymin, ymax = df.x.min(), df.x.max(), df.y.min(), df.y.max()
    for ax, obj in zip(axes.flat, OBJ_COLS):
        z = df[obj].to_numpy(float)
        Zg = df.pivot(index="wb", columns="wa", values=obj)
        floor = FLOORS[obj]
        spacing = (z.max() - z.min()) / 7
        grey = floor is not None and spacing < floor
        cs = ax.contour(np.array(Zg.columns) * NAMEPLATE[a], np.array(Zg.index) * NAMEPLATE[b],
                        Zg.values, levels=7,
                        colors="lightgrey" if grey else "tab:blue",
                        linewidths=0.8 if grey else 1.1)
        if grey:
            ax.annotate("contour spacing\n< noise floor", xy=(0.5, 0.55),
                        xycoords="axes fraction", ha="center", fontsize=9, color="crimson")
        else:
            ax.clabel(cs, fontsize=6, fmt="%.3g")
        for T in np.percentile(df.x + df.y, [20, 40, 60, 80]):
            seg = clip_segment(T, xmin, xmax, ymin, ymax)
            if seg:
                ax.plot([seg[0], seg[1]], [T - seg[0], T - seg[1]], "--",
                        color="grey", lw=0.7, alpha=0.7)
        ax.plot(df.x, df.y, ".", color="k", ms=2, alpha=0.3)
        ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax)
        v = VS[(VS.pair == f"{a}|{b}") & (VS.objective == obj)].iloc[0]
        txt = f"tot {v.r2_tot:.2f} | full {v.r2_full:.2f} | Δloc {v.dloc:.2f} | tilt 2^{v.log2_tilt:.2f}"
        ax.annotate(txt, xy=(0.03, 0.03), xycoords="axes fraction", fontsize=7,
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.85))
        ax.set_xlabel(f"PEM at {a} [MW]"); ax.set_ylabel(f"PEM at {b} [MW]")
        ax.set_title(SHORT[obj], fontsize=10)
    cav = " — nuclear: g1 APPROX caveat (fuel add-back pending PI review)" if "nuclear" in (a, b) else ""
    fig.suptitle(f"{a} × {b} (scenario C, B=40): iso-value contours vs iso-total guides{cav}",
                 fontsize=11)
    fig.tight_layout()
    fig.savefig(f"figs/pair_{a}_{b}_8c.png", dpi=150); plt.close(fig)
print("panel figures written")

panel figures written


## OAT-vs-grid-edge consistency (separate check, never averaged away)

Grid edges hold the partner at ω = 0.05; OAT rows have the partner ABSENT — a real, small
difference. Tolerance = floor + the partner's own ω = 0.05 OAT effect (|f_OAT(0.05) − base|).
OAT curves interpolated to lattice levels where the old per-tier grids differ (pv/tail flagged).

In [6]:
swC_dm = pd.read_csv(os.path.join(CAMPAIGN, "waves", "sweep_C", "design_matrix.csv"))
swC_full = swC_dm.merge(swC[["index"] + OBJ_COLS], on="index", validate="1:1")
bfC_dm = pd.read_csv(os.path.join(CAMPAIGN, "waves", "stage2_backfill_C", "design_matrix.csv"))
bfC_ob = pd.read_csv(os.path.join(CAMPAIGN, "waves", "stage2_backfill_C", "objectives.csv"))
bfC = bfC_dm.merge(bfC_ob[["index"] + OBJ_COLS], on="index", validate="1:1")
oat_pool = pd.concat([swC_full, bfC], ignore_index=True)

def oat_curve(tier, obj):
    m = oat_pool[f"{tier}_omega"].notna()
    for t in TIERS:
        if t != tier:
            m &= oat_pool[f"{t}_omega"].isna()
    sub = oat_pool[m].sort_values(f"{tier}_omega")
    return sub[f"{tier}_omega"].round(10).to_numpy(), sub[obj].to_numpy(float)

edge_rows = []
for (a, b), df in DATA.items():
    for obj in FLOORED:
        floor = FLOORS[obj]
        for tier, partner, wcol, pcol in ((a, b, "wa", "wb"), (b, a, "wb", "wa")):
            wo, yo = oat_curve(tier, obj)
            wp, yp = oat_curve(partner, obj)
            partner_eff = abs(np.interp(0.05, wp, yp) - BASE[obj])
            tol = floor + partner_eff
            edge = df[df[pcol] == 0.05].sort_values(wcol)
            interp_flag = not np.isin(edge[wcol].to_numpy(), np.round(wo, 10)).all()
            oat_at = np.interp(edge[wcol], wo, yo)
            disc = np.abs(edge[obj].to_numpy(float) - oat_at)
            edge_rows.append({"pair": f"{a}|{b}", "edge_tier": tier, "objective": obj,
                              "max_disc": float(disc.max()), "tolerance": tol,
                              "beyond_tol": bool(disc.max() > tol),
                              "oat_interp": interp_flag})
EDGE = pd.DataFrame(edge_rows)
EDGE.to_csv("pair_edge_consistency.csv", index=False)
print(EDGE.round(1).to_string(index=False))
print("\nbeyond-tolerance flags:", int(EDGE.beyond_tol.sum()), "of", len(EDGE),
      "(flagged, never averaged away)")

             pair edge_tier                     objective   max_disc  tolerance  beyond_tol  oat_interp
 nuclear|wind_317   nuclear                 load_shed_mwh     5631.9     4595.7        True       False
 nuclear|wind_317  wind_317                 load_shed_mwh     2874.1     7098.0       False       False
 nuclear|wind_317   nuclear          true_curtailment_mwh    68857.2    78716.1       False       False
 nuclear|wind_317  wind_317          true_curtailment_mwh    34792.7    35871.2       False       False
 nuclear|wind_317   nuclear            total_cost_raw_usd  3840661.2  3308199.5        True       False
 nuclear|wind_317  wind_317            total_cost_raw_usd 25928372.3 26506061.9       False       False
 nuclear|wind_317   nuclear total_cost_less_synthetic_usd  3571944.2  2277089.4        True       False
 nuclear|wind_317  wind_317 total_cost_less_synthetic_usd 26229613.6 26823861.1       False       False
          pv|tail        pv                 load_shed_mwh     32

## Conditional 6-D supplement (pre-registered EXPLORATORY; small-n caveat)

`stage2_C_n0/objectives.csv` exists → the pooled 26-point (16 n₀ + 10 back-fill) LOO check:
GP on total PEM MW alone vs GP on the full 6-D allocation.

In [7]:
n0_dm = pd.read_csv(os.path.join(CAMPAIGN, "waves", "stage2_C_n0", "design_matrix.csv"))
n0_ob = pd.read_csv(os.path.join(CAMPAIGN, "waves", "stage2_C_n0", "objectives.csv"))
n0 = n0_dm.merge(n0_ob[["index"] + OBJ_COLS], on="index", validate="1:1")
pool26 = pd.concat([n0, bfC], ignore_index=True)
Xmw = np.column_stack([pool26[f"{t}_omega"].fillna(0.0) * NAMEPLATE[t] for t in sorted(TIERS)])
Xtot = Xmw.sum(axis=1, keepdims=True)
sup_rows = []
for obj in OBJ_COLS:
    y = pool26[obj].to_numpy(float)
    r_tot, _ = loo_rmse(Xtot, y)
    r_full, _ = loo_rmse(Xmw, y)
    sup_rows.append({"objective": obj, "loo_rmse_total_only": r_tot,
                     "loo_rmse_6d": r_full, "ratio_total/6d": r_tot / r_full})
SUP = pd.DataFrame(sup_rows)
SUP.to_csv("pool26_total_vs_allocation.csv", index=False)
print(SUP.round(3).to_string(index=False))
print("\nEXPLORATORY (n=26 in 6-D): ratio > 1 = allocation carries information beyond total.")

                    objective  loo_rmse_total_only  loo_rmse_6d  ratio_total/6d
                load_shed_mwh             3141.066     2935.879           1.070
         true_curtailment_mwh            98753.672    84988.105           1.162
           total_cost_raw_usd         16874474.039  6415434.952           2.630
total_cost_less_synthetic_usd         16939261.651  6856739.442           2.470
        reserve_shortfall_mwh            13572.744    12973.802           1.046
               thermal_starts              174.965       99.080           1.766

EXPLORATORY (n=26 in 6-D): ratio > 1 = allocation carries information beyond total.


## M-decision machinery: redundancy on the pooled grid designs

Kendall τ on ranks of the pooled 4×81 grid designs + drop-one Pareto sensitivity
(front-overlap ≥ 95% AND hypervolume change < 1% → redundant for MOBO). 19b prior:
cost-vs-{shed, reserve, starts} r ≥ 0.96 cluster.

In [8]:
pool = pd.concat([df.assign(pair=f"{a}|{b}") for (a, b), df in DATA.items()],
                 ignore_index=True)   # 324 designs
tau = pd.DataFrame(index=OBJ_COLS, columns=OBJ_COLS, dtype=float)
for o1, o2 in itertools.combinations_with_replacement(OBJ_COLS, 2):
    t = stats.kendalltau(pool[o1], pool[o2]).statistic
    tau.loc[o1, o2] = tau.loc[o2, o1] = round(float(t), 3)
tau.to_csv("pooled_kendall_tau.csv")
print("Kendall tau (pooled 324 grid designs, minimization sense for all):")
print(tau.to_string())

def nondominated(Y):
    n = len(Y); nd = np.ones(n, bool)
    for i in range(n):
        if nd[i]:
            dom = np.all(Y <= Y[i], axis=1) & np.any(Y < Y[i], axis=1)
            if dom.any():
                nd[i] = False
    return nd

def hv_mc(Y, n_mc=200_000):
    # normalized minimization HV against ref 1.1, MC (seeded)
    r = np.random.default_rng(MC_SEED)
    lo, hi = Y.min(0), Y.max(0)
    Z = (Y - lo) / np.where(hi - lo <= 0, 1, hi - lo)
    S = r.uniform(0, 1.1, size=(n_mc, Y.shape[1]))
    dominated = np.zeros(n_mc, bool)
    for z in Z:
        dominated |= np.all(S >= z, axis=1)
    return dominated.mean() * (1.1 ** Y.shape[1])

Yfull = pool[OBJ_COLS].to_numpy(float)
ndf = nondominated(Yfull); nd_idx = set(np.where(ndf)[0])
hv_full = hv_mc(Yfull[ndf])
drop_rows = []
for k, obj in enumerate(OBJ_COLS):
    cols = [j for j in range(len(OBJ_COLS)) if j != k]
    ndk = nondominated(Yfull[:, cols]); ndk_idx = set(np.where(ndk)[0])
    overlap = len(ndk_idx & nd_idx) / len(nd_idx)
    hv_k = hv_mc(Yfull[list(ndk_idx)][:, :])   # evaluate the reduced front in FULL space
    drop_rows.append({"dropped": obj, "front_overlap": round(overlap, 3),
                      "hv_change_pct": round(100 * (hv_full - hv_k) / hv_full, 2),
                      "redundant_for_MOBO": bool(overlap >= 0.95 and
                                                 abs(hv_full - hv_k) / hv_full < 0.01)})
DROP = pd.DataFrame(drop_rows)
DROP.to_csv("drop_one_pareto.csv", index=False)
print(f"\n|ND(full M=6)| = {len(nd_idx)} of 324; HV_full = {hv_full:.4f}")
print(DROP.to_string(index=False))

Kendall tau (pooled 324 grid designs, minimization sense for all):
                               load_shed_mwh  true_curtailment_mwh  total_cost_raw_usd  total_cost_less_synthetic_usd  reserve_shortfall_mwh  thermal_starts
load_shed_mwh                          1.000                 0.748              -0.590                         -0.561                  0.861           0.839
true_curtailment_mwh                   0.748                 1.000              -0.662                         -0.631                  0.806           0.847
total_cost_raw_usd                    -0.590                -0.662               1.000                          0.959                 -0.645          -0.660
total_cost_less_synthetic_usd         -0.561                -0.631               0.959                          1.000                 -0.615          -0.627
reserve_shortfall_mwh                  0.861                 0.806              -0.645                         -0.615                  1.000        


|ND(full M=6)| = 81 of 324; HV_full = 0.4302
                      dropped  front_overlap  hv_change_pct  redundant_for_MOBO
                load_shed_mwh          0.938           0.05               False
         true_curtailment_mwh          0.815           0.24               False
           total_cost_raw_usd          0.901           0.01               False
total_cost_less_synthetic_usd          0.988           0.00                True
        reserve_shortfall_mwh          0.988           0.00                True
               thermal_starts          0.988           0.01                True


In [9]:
summary = {
    "pairs": [f"{a}|{b}" for (a, b) in DATA],
    "verdict_stats": VS.replace({np.nan: None}).to_dict("records"),
    "gp_gate": GATE.to_dict("records"),
    "isototal_median_ratio": {f"{p}": {o: (float(ok[(ok.pair==p)&(ok.objective==o)].ratio.median())
                                            if len(ok[(ok.pair==p)&(ok.objective==o)]) else None)
                                        for o in FLOORED}
                               for p in ok.pair.unique()},
    "edge_flags": int(EDGE.beyond_tol.sum()),
    "supplement26": SUP.to_dict("records"),
    "kendall_tau": {f"{a}|{b2}": float(tau.loc[a, b2])
                    for a in OBJ_COLS for b2 in OBJ_COLS if a < b2},
    "drop_one": DROP.to_dict("records"),
    "base_anchors": BASE,
}
with open("t2_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("t2_summary.json written; figs:", sorted(f for f in os.listdir('figs') if 'pair' in f))

t2_summary.json written; figs: ['pair_nuclear_pv_8c.png', 'pair_nuclear_wind_317_8c.png', 'pair_pv_tail_8c.png', 'pair_wind_303_wind_317_8c.png']
